In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py

plt.rcParams.update({
    'font.size': 10,
    'font.family': 'serif',
    'mathtext.fontset': 'dejavuserif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 1.2,
    'axes.labelsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi' : 200,
})

In [2]:
import numpy as np
import h5py
import matplotlib.pyplot as plt

def plot_predictions_vs_truth(
    hdf5_file: str,
    confinement_data_file: str,
    psi_target: float = 0.90,           # <-- target ψ here
    label_tolerance_ms: float = 0.6,    # nearest-neighbor tolerance in ms
    block_avg_n: int = 18,              # average this many consecutive outputs
    block_skip_ms: float = 1.0,         # then skip forward this many milliseconds
    exp_alpha: float = 0.001,           # exponential low-pass alpha (tunable)
    colors=('tab:orange','tab:blue'),
    draw_event_vlines: bool = True,
):
    """Plot NN predictions vs. CCTD ground truth at fixed ψ using nearest-neighbor (no interpolation)."""

    # ---- helpers ----
    def _format_psi_key(psi: float) -> str:
        s = f"{psi:.3f}".rstrip('0').rstrip('.')
        return "psi_" + s.replace('.', 'p')
    
    # --- NEW: parse keys like "psi_0p93" -> 0.93 ---
    def _parse_psi_key(key: str) -> float:
        if not key.startswith("psi_"):
            raise ValueError(key)
        return float(key[4:].replace("p", "."))

    # --- NEW: choose closest available ψ for a given shot (tie -> larger ψ) ---
    def _choose_best_available_psi(f_conf, shot_id: str, psi_req: float):
        base = f"{shot_id}/interpolated_psi"
        if base not in f_conf:
            return None, None  # (psi_used, conf_path)

        gpsi = f_conf[base]
        candidates = []
        for k in gpsi.keys():
            if not k.startswith("psi_"):
                continue
            try:
                psi_val = _parse_psi_key(k)
            except Exception:
                continue
            candidates.append((psi_val, k))

        if not candidates:
            return None, None

        # min by (distance, -psi) => tie goes to larger psi
        psi_used, key_used = min(candidates, key=lambda pk: (abs(pk[0] - psi_req), -pk[0]))
        return float(psi_used), f"{base}/{key_used}"

    def _merge_intervals(intervals):
        if not intervals: return []
        ints = sorted((float(a), float(b)) for a, b in intervals)
        merged = [list(ints[0])]
        for a, b in ints[1:]:
            if a <= merged[-1][1]:
                merged[-1][1] = max(merged[-1][1], b)
            else:
                merged.append([a, b])
        return merged

    def _align_truth_to_times(times_ms, label_times_ms, label_values, tol_ms=0.6):
        """Nearest neighbor (no interpolation). NaN if no label time within tol."""
        t  = np.asarray(times_ms, float)
        lt = np.asarray(label_times_ms, float)
        lv = np.asarray(label_values, float)
        if t.size == 0 or lt.size == 0:
            return np.full(t.shape, np.nan, dtype=float)
        idx  = np.searchsorted(lt, t, side='left')
        idx  = np.clip(idx, 0, lt.size-1)
        left = np.clip(idx-1, 0, lt.size-1)
        use_left = np.abs(t - lt[left]) < np.abs(t - lt[idx])
        nn   = np.where(use_left, left, idx)
        out  = lv[nn]
        out[np.abs(t - lt[nn]) > tol_ms] = np.nan
        return out

    def _block_average_and_skip(y, t_ms, avg_n=18, skip_ms=1.0, events=None):
        """
        Average avg_n consecutive samples, then advance start time by skip_ms (relative to last used time).
        Returns (t_out, y_out, events_out).
        """
        y = np.asarray(y, float)
        t = np.asarray(t_ms, float)
        if events is not None:
            events = np.asarray(events)

        if t.size == 0:
            if events is None:
                return t.copy(), y.copy(), None
            return t.copy(), y.copy(), events.copy()

        t_out = []
        y_out = []
        e_out = [] if events is not None else None

        i = 0
        N = t.size

        while i + avg_n <= N:
            sl = slice(i, i + avg_n)

            # average (ignore NaNs if any)
            y_block = y[sl]
            t_block = t[sl]
            y_mean = np.nanmean(y_block)
            t_mean = np.nanmean(t_block)

            t_out.append(t_mean)
            y_out.append(y_mean)

            if events is not None:
                ev_block = events[sl]
                # majority vote (robust for strings)
                try:
                    vals, counts = np.unique(ev_block.astype(str), return_counts=True)
                    e_out.append(vals[int(np.argmax(counts))])
                except Exception:
                    e_out.append(str(ev_block[0]))

            # advance i by time (end of block + skip_ms)
            next_t = t[sl.stop - 1] + float(skip_ms)
            i = int(np.searchsorted(t, next_t, side='left'))

        t_out = np.asarray(t_out, float)
        y_out = np.asarray(y_out, float)
        if e_out is not None:
            e_out = np.asarray(e_out, dtype=object)
        return t_out, y_out, e_out

    def exp_lowpass(y_raw, alpha=0.001):
        """
        Causal exponential low-pass:
        y_f[t] = (1 - alpha) * y_f[t-1] + alpha * y_raw[t]

        NaNs: if y_raw[t] is NaN, we carry forward y_f[t-1].
        Init: first finite sample sets y_f to that value.
        """
        y_raw = np.asarray(y_raw, float)
        y_f = np.full_like(y_raw, np.nan, dtype=float)

        a = float(alpha)
        prev_f = np.nan

        for t, x in enumerate(y_raw):
            if not np.isfinite(x):
                y_f[t] = prev_f
                continue

            if not np.isfinite(prev_f):
                prev_f = x  # initialize
            else:
                prev_f = (1.0 - a) * prev_f + a * x

            y_f[t] = prev_f

        return y_f

    # ---- main ----
    psi_key_req = _format_psi_key(psi_target)

    with h5py.File(hdf5_file, 'r') as f_pred, h5py.File(confinement_data_file, 'r') as f_conf:
        shot_ids = [k for k in f_pred.keys() if not k.startswith('_')]
        if not shot_ids:
            print("No shot groups found in predictions file.")
            return

        for shot_id in shot_ids:
            g = f_pred[shot_id]
            if not all(k in g for k in ("predictions", "times_ms")):
                print(f"Skipping shot {shot_id}: missing predictions/times_ms.")
                continue

            preds = np.asarray(g["predictions"][()], float)
            if preds.ndim == 2:   # e.g., (N, 4) from old runs
                preds = preds[:, 0]

            t_ms  = np.asarray(g["times_ms"][()], float)

            events = None
            if "events" in g:
                try:
                    events = g["events"][()].astype(str)
                except Exception:
                    events = None

            # sort & (optional) simple decimation first
            order = np.argsort(t_ms)
            t_ms, preds = t_ms[order], preds[order]
            if events is not None:
                events = events[order]

            # --- NEW: block average + time skip ---
            t_ms2, preds2, events2 = _block_average_and_skip(
                preds, t_ms,
                avg_n=block_avg_n,
                skip_ms=block_skip_ms,
                events=events
            )

            # --- UPDATED: pick best available psi group if requested key missing ---
            conf_path_req = f"{shot_id}/interpolated_psi/{psi_key_req}"
            if conf_path_req in f_conf:
                conf_path = conf_path_req
                psi_used = float(psi_target)
            else:
                psi_used, conf_path = _choose_best_available_psi(f_conf, shot_id, psi_target)
                if conf_path is None:
                    print(f"Shot {shot_id}: no interpolated_psi entries found in confinement file.")
                    continue
                print(f"Shot {shot_id}: requested ψ={psi_target:.3f} not found; using nearest ψ={psi_used:.3f} ({conf_path.split('/')[-1]}).")

            # pull truth at ψ directly from confinement file (no interpolation),
            # then align to *new* t_ms2 via nearest neighbor within tolerance
            # conf_path = f"{shot_id}/interpolated_psi/{psi_key}"
            # if conf_path not in f_conf:
            #     print(f"Shot {shot_id}: {conf_path} not found in confinement file. trying {psi_key} ")
            #     continue

            lt = np.asarray(f_conf[conf_path]["label_times"][()], float)  # ms
            lv = np.asarray(f_conf[conf_path]["vZ"][()], float)           # same length

            trues2 = _align_truth_to_times(t_ms2, lt, lv, tol_ms=label_tolerance_ms)

            # --- NEW: exponential low-pass ---
            preds_f = exp_lowpass(preds2, alpha=exp_alpha)
            # trues_f = exp_lowpass(trues2, alpha=exp_alpha)
            trues_f = trues2

            # figure
            fig, ax = plt.subplots(1, 1, figsize=(10, 3))
            fig.suptitle(f"Shot #{shot_id}  |  ψ={psi_target:.2f}", fontsize=12)

            # event boundaries (based on block-reduced events)
            if draw_event_vlines and events2 is not None and events2.size > 1:
                change = np.where(events2[1:] != events2[:-1])[0]
                for idx in change:
                    ax.axvline(t_ms2[idx], color='0.7', linestyle='--', linewidth=1)

            # plot (keep your sign convention)
            ax.plot(t_ms2, np.abs(preds_f), label="LP NN result", lw=2.0, color=colors[0], zorder=0, alpha=0.8)
            if np.isfinite(trues_f).any():
                ax.plot(t_ms2, np.abs(trues_f), label="CCTD ground truth", linestyle="--", lw=1.5, color=colors[1], zorder=1)
            else:
                print(f"Shot {shot_id}: aligned truth all NaN (check tolerance {label_tolerance_ms} ms).")

            ax.set_ylabel(r"$|v^{BES}_{\theta}|$ (km/s)")
            ax.set_xlabel("Time (ms)")

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            ax.legend(by_label.values(), by_label.keys(), fontsize="small", loc="best")

            ax.grid(True, alpha=0.25)
            plt.tight_layout()
            plt.show()

In [ ]:
line_colors = ["#7570b3", "#009E73", "#d95f02",]
# run_no = 47160693
run_no = 47900264
# run_no = 'ft_47897332'
# run_no = 'ft_47958097'


# confinement_data_file = "/pscratch/sd/k/kevinsg/bes_ml_jobs/confinement_data/20251027_raw_signals_psi_interp.hdf5"
# confinement_data_file = "/pscratch/sd/k/kevinsg/bes_ml_jobs/confinement_data/205867_psi_interp.hdf5"
# confinement_data_file = "/pscratch/sd/k/kevinsg/bes_ml_jobs/confinement_data/jan15_psi_interp.hdf5"
confinement_data_file = "/pscratch/sd/k/kevinsg/bes_ml_jobs/confinement_data/jan18_psi_interp_merged.hdf5"
# hdf5_path = f"/pscratch/sd/k/kevinsg/bes_ml_jobs/exp_gill01/{run_no}/velo_predictions_1.hdf5"
# hdf5_path = f"/global/cfs/cdirs/m3586/kgill/bes-ml/bes_ml2/exp_gill01/{run_no}/predictions_0.hdf5"
# plot_predictions_vs_truth(
#     hdf5_file=hdf5_path,
#     confinement_data_file = confinement_data_file,       
#     psi_target = 0.92,
#     block_avg_n = 18,              # average this many consecutive outputs
#     block_skip_ms = 1.0,         # then skip forward this many milliseconds
#     exp_alpha = 0.1,  
#     colors=line_colors,
#     draw_event_vlines=True)

for pred_idx in range(24):
    hdf5_path = f"/pscratch/sd/k/kevinsg/bes_ml_jobs/exp_gill01/{run_no}/predictions_{pred_idx}.hdf5"
    try:
        plot_predictions_vs_truth(
            hdf5_file=hdf5_path,
            confinement_data_file = confinement_data_file,       
            psi_target = 0.92,
            block_avg_n = 18,              # average this many consecutive outputs
            block_skip_ms = 1.0,         # then skip forward this many milliseconds
            exp_alpha = 0.1,  
            colors=line_colors,
            draw_event_vlines=True)
    except FileNotFoundError:
        print(f"missing predictions_{pred_idx} file")

In [18]:
def plot_elm_predictions_vs_truth(
    hdf5_file: str,
    block_avg_n: int = 18,              # average this many consecutive outputs
    block_skip_ms: float = 1.0,         # then skip forward this many milliseconds
    exp_alpha: float = 0.001,           # exponential low-pass alpha
    draw_event_vlines: bool = True,
    apply_sigmoid: bool = False,        # set True if "predictions" are logits and you want prob
    clip_prob: bool = True,             # clip to [0,1] after sigmoid (or even without)
    colors=("tab:orange", "tab:blue"),
    max_points_no_block: int = 300_000, # safety if you disable block averaging
):
    """
    Plot ELM predictions vs truth from HDF5 with structure:
      <shot>/predictions   (N,)
      <shot>/true_labels   (N,)
      <shot>/times_ms      (N,)
      <shot>/events        (N,) optional
    """

    # ---------- helpers ----------
    def _block_average_and_skip(y, t_ms, avg_n=18, skip_ms=1.0, events=None):
        """
        Average avg_n consecutive samples, then advance start time by skip_ms (relative to last used time).
        Returns (t_out, y_out, events_out).
        """
        y = np.asarray(y, float)
        t = np.asarray(t_ms, float)
        if events is not None:
            events = np.asarray(events)

        if t.size == 0:
            if events is None:
                return t.copy(), y.copy(), None
            return t.copy(), y.copy(), events.copy()

        t_out, y_out = [], []
        e_out = [] if events is not None else None

        i, N = 0, t.size
        while i + avg_n <= N:
            sl = slice(i, i + avg_n)

            y_block = y[sl]
            t_block = t[sl]

            y_mean = np.nanmean(y_block)
            t_mean = np.nanmean(t_block)

            t_out.append(t_mean)
            y_out.append(y_mean)

            if events is not None:
                ev_block = events[sl]
                # majority vote (robust for strings/objects)
                vals, counts = np.unique(ev_block.astype(str), return_counts=True)
                e_out.append(vals[int(np.argmax(counts))])

            next_t = t[sl.stop - 1] + float(skip_ms)
            i = int(np.searchsorted(t, next_t, side="left"))

        t_out = np.asarray(t_out, float)
        y_out = np.asarray(y_out, float)
        if e_out is not None:
            e_out = np.asarray(e_out, dtype=object)
        return t_out, y_out, e_out

    def exp_lowpass(y_raw, alpha=0.001):
        """
        Causal exponential low-pass:
          y_f[t] = (1-alpha)*y_f[t-1] + alpha*y_raw[t]
        NaNs: carry forward previous filtered value.
        """
        y_raw = np.asarray(y_raw, float)
        y_f = np.full_like(y_raw, np.nan, dtype=float)

        a = float(alpha)
        prev_f = np.nan

        for i, x in enumerate(y_raw):
            if not np.isfinite(x):
                y_f[i] = prev_f
                continue
            if not np.isfinite(prev_f):
                prev_f = x
            else:
                prev_f = (1.0 - a) * prev_f + a * x
            y_f[i] = prev_f

        return y_f

    def _sigmoid(x):
        x = np.asarray(x, float)
        # stable sigmoid
        out = np.empty_like(x, dtype=float)
        pos = x >= 0
        out[pos] = 1.0 / (1.0 + np.exp(-x[pos]))
        ex = np.exp(x[~pos])
        out[~pos] = ex / (1.0 + ex)
        return out

    # ---------- main ----------
    with h5py.File(hdf5_file, "r") as f:
        shot_ids = [k for k in f.keys() if not k.startswith("_")]
        if not shot_ids:
            print("No shot groups found in predictions file.")
            return

        for shot_id in shot_ids:
            g = f[shot_id]
            req = ("predictions", "true_labels", "times_ms")
            if not all(k in g for k in req):
                print(f"Skipping shot {shot_id}: missing one of {req}.")
                continue

            preds = np.asarray(g["predictions"][()], float)
            truth = np.asarray(g["true_labels"][()], float)
            t_ms  = np.asarray(g["times_ms"][()], float)
            print("pred raw mean/std:", np.nanmean(preds), np.nanstd(preds), "min/max:", np.nanmin(preds), np.nanmax(preds))
            print("truth mean/std:   ", np.nanmean(truth), np.nanstd(truth))


            events = None
            if "events" in g:
                try:
                    events = g["events"][()].astype(str)
                except Exception:
                    # if already object
                    events = np.asarray(g["events"][()], dtype=object).astype(str)

            # sort by time (critical)
            order = np.argsort(t_ms)
            t_ms, preds, truth = t_ms[order], preds[order], truth[order]
            if events is not None:
                events = events[order]

            # if not doing block averaging, prevent plotting absurdly huge arrays by decimating
            if (block_avg_n is None) or (block_avg_n <= 1):
                if t_ms.size > max_points_no_block:
                    stride = int(np.ceil(t_ms.size / max_points_no_block))
                    t_ms, preds, truth = t_ms[::stride], preds[::stride], truth[::stride]
                    if events is not None:
                        events = events[::stride]

                t_ms2, preds2, events2 = t_ms, preds, events
                truth2 = truth
            else:
                # block average + skip for preds and truth (and events)
                t_ms2, preds2, events2 = _block_average_and_skip(
                    preds, t_ms, avg_n=int(block_avg_n), skip_ms=float(block_skip_ms), events=events
                )
                # for truth, use the same reduction windows by re-running on truth with same events
                t_ms2b, truth2, _ = _block_average_and_skip(
                    truth, t_ms, avg_n=int(block_avg_n), skip_ms=float(block_skip_ms), events=events
                )
                # they should match; if tiny float diffs occur, just trust t_ms2
                if t_ms2b.size != t_ms2.size:
                    # fallback: nearest-neighbor truth onto t_ms2
                    tt = np.asarray(t_ms, float)
                    yy = np.asarray(truth, float)
                    idx = np.searchsorted(tt, t_ms2, side="left")
                    idx = np.clip(idx, 0, tt.size - 1)
                    left = np.clip(idx - 1, 0, tt.size - 1)
                    use_left = np.abs(t_ms2 - tt[left]) < np.abs(t_ms2 - tt[idx])
                    nn = np.where(use_left, left, idx)
                    truth2 = yy[nn]

            # optional sigmoid (if preds are logits)
            preds_plot = _sigmoid(preds2) if apply_sigmoid else np.asarray(preds2, float)
            truth_plot = np.asarray(truth2, float)

            if clip_prob:
                preds_plot = np.clip(preds_plot, 0.0, 1.0)
                truth_plot = np.clip(truth_plot, 0.0, 1.0)

            # low-pass (causal)
            preds_f = exp_lowpass(preds_plot, alpha=exp_alpha)
            # truth usually doesn't need filtering; keep raw unless you want it too
            truth_f = truth_plot

            # ----- plot -----
            fig, ax = plt.subplots(1, 1, figsize=(10, 3))
            title_bits = [f"Shot #{shot_id}", "ELM prediction"]
            if apply_sigmoid:
                title_bits.append("(sigmoid)")
            fig.suptitle("  |  ".join(title_bits), fontsize=12)

            # event boundaries
            if draw_event_vlines and (events2 is not None) and (len(events2) > 1):
                change = np.where(events2[1:] != events2[:-1])[0]
                for j in change:
                    ax.axvline(t_ms2[j], color="0.7", linestyle="--", linewidth=1)

            ax.plot(t_ms2, preds_f, label="LP NN prediction", lw=2.0, color=colors[0], alpha=0.85)
            if np.isfinite(truth_f).any():
                ax.plot(t_ms2, truth_f, label="True label", lw=4.5, color=colors[1], alpha=0.9)
            else:
                print(f"Shot {shot_id}: true_labels all NaN?")

            ax.set_xlabel("Time (ms)")
            ax.set_ylabel("P(ELM)")
            ax.set_ylim(-0.05, 1.05) if clip_prob else None
            ax.grid(True, alpha=0.25)
            ax.legend(loc="best", fontsize="small")

            plt.tight_layout()
            plt.show()


In [ ]:
run_id = 'ft_47958097'
file = f"/pscratch/sd/k/kevinsg/bes_ml_jobs/exp_gill01/{run_id}/predictions.hdf5"
line_colors = ["#7570b3", "#009E73", "#d95f02",]

plot_elm_predictions_vs_truth(
    hdf5_file=file,
    block_avg_n= 18,              # average this many consecutive outputs
    block_skip_ms = 1.0,         # then skip forward this many milliseconds
    exp_alpha = 1.0,           # exponential low-pass alpha
    draw_event_vlines = True,
    apply_sigmoid = False,        # set True if "predictions" are logits and you want prob
    clip_prob = True,             # clip to [0,1] after sigmoid (or even without)
    colors=line_colors,
    max_points_no_block = 300_000, # safety if you disable block averaging
)